In [ ]:
# Imports

import os
from pathlib import Path
from dotenv import load_dotenv
from IPython.display import Markdown, display
from openai import OpenAI

# Connect to OpenAI

load_dotenv(override=True)
api_key = os.getenv("OPENAI_API_KEY")

if not api_key:
    print("No API key found - please check your .env file")

openai = OpenAI()

# Step 1: Set up the folders

LOGS_DIR = Path("logs")        # drop new .log / .txt files here
TICKETS_DIR = Path("tickets")  # generated tickets go here
LOGS_DIR.mkdir(exist_ok=True)
TICKETS_DIR.mkdir(exist_ok=True)

# Step 2: Create your prompts

system_prompt = """
You are a senior QA engineer who turns raw test failure logs into clear bug tickets.

Respond in markdown with exactly these sections:
- **Title**: one line, specific enough to find in a bug tracker
- **Summary**: no more than two sentences
- **Severity**: Critical, High, Medium or Low, with a one-line reason
- **Steps to Reproduce**: a numbered list
- **Expected Result**
- **Actual Result**
- **Likely Component**

Use only the information in the log. Do not invent details.
If information for a section is missing, write "Not provided".
Do not wrap the markdown in a code block.
"""

def user_prompt_for(log_text):
    return "Create a bug ticket from this test log.\n\n" + log_text

# Step 3: Call OpenAI for one log

def create_ticket(log_text):
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_for(log_text)},
    ]
    response = openai.chat.completions.create(
        model="gpt-6-luna",
        reasoning_effort="medium",
        messages=messages,
    )
    return response.choices[0].message.content

# Step 4: Create a ticket for every log that doesn't have one yet
# (to regenerate a ticket, delete it from the tickets folder and run again)

new_tickets = 0

for log_file in sorted(LOGS_DIR.iterdir()):
    if log_file.suffix not in (".log", ".txt"):
        continue
    ticket_file = TICKETS_DIR / f"{log_file.stem}.md"
    if ticket_file.exists():
        continue
    ticket = create_ticket(log_file.read_text(encoding="utf-8"))
    ticket_file.write_text(ticket, encoding="utf-8")
    new_tickets += 1
    display(Markdown(f"### {log_file.name} → {ticket_file}\n\n{ticket}"))

if new_tickets == 0:
    print("No new logs - every log already has a ticket.")